In [1]:
import os
from pathlib import Path

DATA_ROOT = Path.home() / "mne_data"

os.environ["MNE_DATA"] = str(DATA_ROOT)
os.environ["MNE_DATASETS_BNCI_PATH"] = str(DATA_ROOT)
os.environ["MNE_DATASETS_ZHOU2016_PATH"] = str(DATA_ROOT)
os.environ["MOABB_DOWNLOAD_PROVIDER"] = "upstream"

EXPERIMENT_DIR = Path("/Users/liza/Projects/vkr/mi-ood")
CACHE_DIR = EXPERIMENT_DIR / "cache"
RESULTS_DIR = EXPERIMENT_DIR / "results"

print(EXPERIMENT_DIR)

/Users/liza/Projects/vkr/mi-ood


In [3]:
from methods import ShallowOOD
from pipelines import mi_pipelines, ood_pipelines
from processing import *

In [4]:
experiment_mi = "ws_mi_7ch"
experiment_ood = "ws_ood_7ch"
COMMON_CHANNELS = ["FC3", "FCz", "FC4", "C3", "C4", "CP3", "CP4"]
datasets = [ "BNCI2014_001", "Zhou2016", "Wang2025", "Yang2025_3C", "Zhou2020" ]


In [5]:
for dataset in datasets:
    print( dataset )
    subjects = get_subjects(dataset) 

    for subject in subjects:
        data = None
        for method, pipeline in mi_pipelines.items():
            result = {"dataset": dataset, "subject": subject, "experiment": experiment_mi, "method": method}
            if result_path(result).exists():
                continue
            if data is None:
                data = load_subject_data(dataset, subject, channels=COMMON_CHANNELS, within_session=True)
            train_data, val_data, test_data = data
            result["MI_BA"], result["MI_AUROC"] = MI_metrics(train_data, test_data, pipeline)
            save_result(result)
            print(f"{dataset} s{subject} {method}: {result['MI_BA']:.3f} {result['MI_AUROC']:.3f}")

        data = None
        shallow = None
        for method, pipeline in ood_pipelines.items():
            result = {"dataset": dataset, "subject": subject, "experiment": experiment_ood, "method": method}
            if result_path(result).exists():
                continue
            if data is None:
                data = load_subject_data(dataset, subject, channels=COMMON_CHANNELS, include_feet=True, within_session=True)
                train_data, val_data, test_data = data
            if isinstance(pipeline, ShallowOOD):
                if shallow is None:
                    shallow = train(train_data, mi_pipelines["ShallowConvNet"])

                model = clone(pipeline).fit(*train_data, classifier=shallow)
            else:
                model = train(train_data, pipeline)

            result["OOD_AUROC"] = OOD_AUROC(test_data, model)
            save_result(result)
            print(f"{dataset} s{subject} {method}: {result['OOD_AUROC']:.3f}")

    display(summarize_results(dataset, experiment_mi).round(3))
    display(summarize_results(dataset, experiment_mi, metric="MI_AUROC").round(3))
    display(summarize_results(dataset, experiment_ood, metric="OOD_AUROC").round(3))

BNCI2014_001


,metric,n
method,,
ACM+TS+SVM,0.735,9
EEGNet,0.741,9
FBCSP+ElasticNet,0.750,9
Log-bandpower+shLDA,0.648,9
Microstates+GB,0.679,9
ShallowConvNet,0.710,9
VAR+shLDA,0.676,9


,metric,n
method,,
ACM+TS+SVM,0.806,9
EEGNet,0.776,9
FBCSP+ElasticNet,0.821,9
Log-bandpower+shLDA,0.683,9
Microstates+GB,0.718,9
ShallowConvNet,0.777,9
VAR+shLDA,0.747,9


,metric,n
method,,
ACM+TS+OCSVM,0.636,9
EEGNet+TempDens,0.751,9
FBCSP+IsolationForest,0.697,9
Log-bandpower+Mahalanobis,0.528,9
Microstates+LOF,0.525,9
ShallowConvNet+kNN,0.708,9
VAR+Mahalanobis,0.550,9


Zhou2016


,metric,n
method,,
ACM+TS+SVM,0.852,4
EEGNet,0.870,4
FBCSP+ElasticNet,0.894,4
Log-bandpower+shLDA,0.825,4
Microstates+GB,0.841,4
ShallowConvNet,0.773,4
VAR+shLDA,0.685,4


... (10 outputs left out)

Zhou2020


,metric,n
method,,
ACM+TS+SVM,0.608,20
EEGNet,0.602,20
FBCSP+ElasticNet,0.602,20
Log-bandpower+shLDA,0.574,20
Microstates+GB,0.542,20
ShallowConvNet,0.598,20
VAR+shLDA,0.540,20


,metric,n
method,,
ACM+TS+SVM,0.644,20
EEGNet,0.622,20
FBCSP+ElasticNet,0.630,20
Log-bandpower+shLDA,0.590,20
Microstates+GB,0.576,20
ShallowConvNet,0.634,20
VAR+shLDA,0.561,20


,metric,n
method,,
ACM+TS+OCSVM,0.557,20
EEGNet+TempDens,0.642,20
FBCSP+IsolationForest,0.629,20
Log-bandpower+Mahalanobis,0.570,20
Microstates+LOF,0.546,20
ShallowConvNet+kNN,0.636,20
VAR+Mahalanobis,0.578,20


In [6]:
from pathlib import Path
import pandas as pd

MI_METHODS = mi_pipelines.keys()
OOD_METHODS = ood_pipelines.keys()

parts = []
for task, experiment, metric in [
    ("MI BA", experiment_mi, "MI_BA"),
    ("MI AUROC", experiment_mi, "MI_AUROC"),
    ("OOD", experiment_ood, "OOD_AUROC") ]:
    d = pd.concat([pd.read_csv(p) for p in (Path("results") / experiment).glob("*.csv")], ignore_index=True)
    methods = OOD_METHODS if task == "OOD" else MI_METHODS
    d = d[d["method"].isin(methods)]
    parts.append(d.assign(task=task, score=d[metric]))
r = pd.concat(parts, ignore_index=True)
means = r.groupby(["task", "method", "dataset"])["score"].mean().unstack("dataset")
table = means.join(r.groupby(["task", "method"])["score"].agg(mean_subjects="mean", sd="std", n="count"))
table["mean_datasets"] = means.mean(axis=1)
display(table.sort_values(["task", "mean_subjects"], ascending=[True, False]).round(3))


BNCI2014_001    TTK  Wang2025  \
task     method                                                     
MI AUROC EEGNet                            0.776  0.658     0.821   
         ACM+TS+SVM                        0.806  0.773     0.753   
         FBCSP+ElasticNet                  0.821  0.804     0.763   
         ShallowConvNet                    0.777  0.646     0.776   
         Log-bandpower+shLDA               0.683  0.750     0.660   
         VAR+shLDA                         0.747  0.648     0.657   
         Microstates+GB                    0.718  0.638     0.656   
MI BA    EEGNet                            0.741  0.633     0.743   
         FBCSP+ElasticNet                  0.750  0.721     0.709   
         ACM+TS+SVM                        0.735  0.735     0.693   
         ShallowConvNet                    0.710  0.604     0.720   
         Log-bandpower+shLDA               0.648  0.700     0.603   
         Microstates+GB                    0.679  0.646     0.612   
         VAR+shLDA                         0.676  0.622     0.609   
OOD      EEGNet+TempDens                   0.751  0.748     0.726   
         ShallowConvNet+kNN                0.708  0.762     0.723   
         FBCSP+IsolationForest             0.697  0.761     0.710   
         ACM+TS+OCSVM                      0.636  0.688     0.656   
         Log-bandpower+Mahalanobis         0.528  0.654     0.612   
         VAR+Mahalanobis                   0.550  0.661     0.564   
         Microstates+LOF                   0.525  0.583     0.596   

                                    Yang2025_3C  Zhou2016  Zhou2020  \
task     method                                                       
MI AUROC EEGNet                           0.694     0.961     0.622   
         ACM+TS+SVM                       0.656     0.928     0.644   
         FBCSP+ElasticNet                 0.632     0.943     0.630   
         ShallowConvNet                   0.692     0.824     0.634   
         Log-bandpower+shLDA              0.550     0.923     0.590   
         VAR+shLDA                        0.625     0.754     0.561   
         Microstates+GB                   0.564     0.875     0.576   
MI BA    EEGNet                           0.651     0.870     0.602   
         FBCSP+ElasticNet                 0.595     0.894     0.602   
         ACM+TS+SVM                       0.622     0.852     0.608   
         ShallowConvNet                   0.660     0.773     0.598   
         Log-bandpower+shLDA              0.511     0.825     0.574   
         Microstates+GB                   0.547     0.841     0.542   
         VAR+shLDA                        0.609     0.685     0.540   
OOD      EEGNet+TempDens                  0.712     0.896     0.642   
         ShallowConvNet+kNN               0.686     0.772     0.636   
         FBCSP+IsolationForest            0.673     0.818     0.629   
         ACM+TS+OCSVM                     0.625     0.776     0.557   
         Log-bandpower+Mahalanobis        0.619     0.725     0.570   
         VAR+Mahalanobis                  0.575     0.685     0.578   
         Microstates+LOF                  0.618     0.703     0.546   

                                    mean_subjects     sd   n  mean_datasets  
task     method                                                              
MI AUROC EEGNet                             0.726  0.205  65          0.755  
         ACM+TS+SVM                         0.723  0.176  65          0.760  
         FBCSP+ElasticNet                   0.723  0.180  65          0.765  
         ShallowConvNet                     0.709  0.166  65          0.725  
         Log-bandpower+shLDA                0.648  0.153  65          0.693  
         VAR+shLDA                          0.639  0.141  65          0.665  
         Microstates+GB                     0.636  0.164  65          0.671  
MI BA    EEGNet                             0.681  0.173  65          0.706  
         FBCSP+ElasticNet         